# Phase 2 — Schema Drift / Quarantine Integration QA

**Safety:** Uses copies of the current production transformation definitions but routes **all writes** to `github_health.qa_drift_*` tables. The QA input is synthetic and stored in `_qa_tests/`. Does **not** execute `driver_session` Cell 6 or modify production Bronze/Silver tables.

**Run order:** all cells from top to bottom. The FULL and INCREMENTAL checks are separate so you can stop and review between them. Keep in the same Databricks folder as `schemas` and `common`.

**Tested conceptually/static-checked only:** run results must be verified in Databricks Serverless.

In [0]:
%run ./schemas

In [0]:
%run ./common

In [0]:
# Dedicated, isolated QA destinations. DO NOT run this in driver_session.
# Same source-field contract and transformation semantics as the active Cell 5.
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CATALOG = "github_health"
QA_BRONZE = f"{CATALOG}.qa_drift_bronze.issues"
QA_SILVER = f"{CATALOG}.qa_drift_silver.issues"
QA_BRONZE_Q = f"{CATALOG}.qa_drift_ops.bronze_quarantine"
QA_SILVER_Q = f"{CATALOG}.qa_drift_ops.silver_quarantine"
LOG_TABLE = f"{CATALOG}.qa_drift_ops.pipeline_execution_logs"
QA_ROOT = Path("/Volumes/github_health/raw/bronze_data/_qa_tests/phase2_drift_integration")

for schema in ("qa_drift_bronze", "qa_drift_silver", "qa_drift_ops"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

for name, schema in [(QA_BRONZE, BRONZE_SCHEMA["issues"]),
                     (QA_SILVER, SILVER_SCHEMA["issues"])]:
    if not spark.catalog.tableExists(name):
        spark.createDataFrame([], schema).write.format("delta").saveAsTable(name)

# Build the QA audit table from the REAL current production log schema.
# This automatically includes recently added inserted_rows / updated_rows.
if not spark.catalog.tableExists(LOG_TABLE):
    spark.createDataFrame([], spark.table(f"{CATALOG}.ops.pipeline_execution_logs").schema) \
        .write.format("delta").saveAsTable(LOG_TABLE)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QA_BRONZE_Q} (
  quarantine_id STRING, batch_id STRING, repository STRING, entity STRING,
  source_file STRING, raw_payload STRING, error_type STRING,
  error_reason STRING, load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
) USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QA_SILVER_Q} (
  quarantine_id STRING, batch_id STRING, repository STRING, entity STRING,
  record_key STRING, raw_payload STRING, error_type STRING,
  error_reason STRING, load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
) USING DELTA
""")

# The actual common.quarantine function uses an ops destination.
# This local wrapper keeps its MERGE semantics, but writes ONLY to QA ops.
def quarantine(spark, df, layer):
    table = QA_BRONZE_Q if layer == "bronze" else QA_SILVER_Q
    if not df.take(1):
        return 0
    df = df.dropDuplicates(["quarantine_id"])
    n = df.count()
    values = {column: f"s.`{column}`" for column in df.columns}
    (DeltaTable.forName(spark, table).alias("t")
      .merge(df.alias("s"), "t.quarantine_id = s.quarantine_id")
      .whenNotMatchedInsert(values=values)
      .execute())
    return n

print("QA destination verification:")
print("Bronze:", QA_BRONZE)
print("Silver:", QA_SILVER)
print("Quarantine:", QA_BRONZE_Q, QA_SILVER_Q)
print("Audit:", LOG_TABLE)
print("Production destinations are NOT used by this QA notebook.")

In [0]:
# ============================================================

# CELL 5: BRONZE AND SILVER TRANSFORMATION FUNCTIONS

# Supports:

# 1. Commits

# 2. Issues

# 3. Pull Requests

# 4. Repository Metadata

# 5. Contributors

#

# This cell loads a read-only source-field contract and defines functions.

# It does NOT process records or write to Delta tables.

# ============================================================

import time

import uuid

import traceback
import json
from pathlib import Path

from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ============================================================

# 1. BUSINESS COLUMNS FOR CHANGE DETECTION

# ============================================================

# These fields are used to calculate payload_hash.

# Exclude batch IDs and ingestion timestamps because they

# are pipeline metadata, not business-data changes.

BUSINESS_COLS = {

    "commits": [

        "sha", "node_id", "commit",

        "author", "committer", "html_url"

    ],

    "issues": [

        "id", "number", "node_id", "title",

        "state", "created_at", "updated_at",

        "closed_at", "user", "labels", "html_url"

    ],

    "pulls": [

        "id", "number", "node_id", "title",

        "state", "draft", "created_at",

        "updated_at", "closed_at", "merged_at",

        "user", "merged", "html_url"

    ],

    "repo_metadata": [

        "id", "name", "full_name", "description",

        "html_url", "language", "default_branch",

        "stargazers_count", "forks_count",

        "open_issues_count", "subscribers_count",

        "archived", "fork", "created_at",

        "updated_at", "pushed_at", "topics", "license"

    ],

    "contributors": [

        "id", "type", "site_admin", "contributions"

    ]

}

# Repository Metadata and Contributors are snapshots.

# Their update order is based on collection time.

SNAPSHOT_ENTITIES = {"repo_metadata", "contributors"}

# Fixed contract of approved top-level GitHub JSON fields.
# This is read only; it is never learned or extended during a pipeline run.
SOURCE_CONTRACT_PATH = Path(
    "/Volumes/github_health/raw/bronze_data/bronze/"
    "_contracts/known_source_fields_v1.json"
)

with SOURCE_CONTRACT_PATH.open("r", encoding="utf-8") as contract_file:
    SOURCE_CONTRACT = json.load(contract_file)

ALLOWED_SOURCE_FIELDS = SOURCE_CONTRACT["allowed_fields"]
required_entities = set(RAW_SCHEMAS)
missing_contracts = required_entities - set(ALLOWED_SOURCE_FIELDS)

if missing_contracts:
    raise ValueError(
        f"Missing source field contracts: {sorted(missing_contracts)}"
    )

for source_entity in required_entities:
    approved_fields = ALLOWED_SOURCE_FIELDS[source_entity]
    if not isinstance(approved_fields, list) or not approved_fields:
        raise ValueError(
            f"Empty or invalid approved field list: {source_entity}"
        )

print("Source field contract loaded for:", sorted(required_entities))

# ============================================================

# 2. MERGE FOR COMMITS, ISSUES AND PULL REQUESTS

# ============================================================

def merge_delta_changed(df, table_name):

    """

    Insert new records or update existing records only when

    the business payload changed and the source timestamp

    is newer than the stored timestamp.

    Prevents older historical data from overwriting newer data.

    """

    columns = df.columns

    # Values used for inserting new records

    values = {

        column: f"s.`{column}`"

        for column in columns

    }

    # Update all supplied fields except the logical primary key

    updates = {

        column: expression

        for column, expression in values.items()

        if column != "record_key"

    }

    (

        DeltaTable.forName(spark, table_name)

        .alias("t")

        .merge(

            df.alias("s"),

            "t.record_key = s.record_key"

        )

        .whenMatchedUpdate(

            condition=(

                "NOT (t.payload_hash <=> s.payload_hash) "

                "AND s.source_updated_at IS NOT NULL "

                "AND (t.source_updated_at IS NULL "

                "OR s.source_updated_at > t.source_updated_at)"

            ),

            set=updates

        )

        .whenNotMatchedInsert(values=values)

        .execute()

    )

    # Get the actual insert/update counts from Delta history

    metrics = spark.sql(

        f"DESCRIBE HISTORY {table_name} LIMIT 1"

    ).first()["operationMetrics"]

    inserted = int(metrics.get("numTargetRowsInserted", 0))

    updated = int(metrics.get("numTargetRowsUpdated", 0))

    return inserted, updated

# ============================================================

# 3. SNAPSHOT-AWARE MERGE

# ============================================================

def merge_delta_metadata_snapshot(

    df, table_name, snapshot_column

):

    """

    Reuse the snapshot-aware MERGE in the common notebook.

    Used for both Repository Metadata and Contributors.

    It uses ingested_at in Bronze and snapshot_at in Silver

    instead of relying on GitHub's updated_at field.

    """

    merge_metadata_snapshot(

        spark,

        df,

        table_name,

        snapshot_column

    )

    # Retrieve MERGE operation statistics

    metrics = spark.sql(

        f"DESCRIBE HISTORY {table_name} LIMIT 1"

    ).first()["operationMetrics"]

    inserted = int(metrics.get("numTargetRowsInserted", 0))

    updated = int(metrics.get("numTargetRowsUpdated", 0))

    return inserted, updated

# ============================================================

# 4. GENERATE LOGICAL RECORD KEYS

# ============================================================

def source_keys_from_raw(raw, repo, entity):

    """

    Generate keys belonging to the current Raw batch.

    Commits use SHA.

    All other entities use GitHub numeric IDs.

    The repository name is included to distinguish records

    belonging to different repositories.

    """

    # Identify the unique source ID

    id_col = (

        F.col("sha")

        if entity == "commits"

        else F.col("id").cast("string")

    )

    # Exclude malformed rows, missing IDs and wrong repositories

    good = (

        raw

        .filter(F.col("_corrupt_record").isNull())

        .filter(

            id_col.isNotNull()

            & (F.trim(id_col) != "")

        )

        .filter(

            F.col("_repo").isNull()

            | (F.col("_repo") == repo)

        )

    )

    # GitHub's Issues API can also return pull requests

    if entity == "issues":

        good = good.filter(

            F.col("pull_request").isNull()

        )

    # Do not pass invalid snapshot keys into Silver

    if entity in SNAPSHOT_ENTITIES:

        good = good.filter(

            F.expr(

                "try_cast(_ingested_at as timestamp)"

            ).isNotNull()

        )

    # Logical key: repository + source record ID

    return good.select(

        F.concat_ws(

            "|",

            F.lit(repo),

            id_col

        ).alias("record_key")

    ).distinct()

# ============================================================
# 5. SCHEMA-DRIFT-AWARE RAW JSONL READER
# ============================================================

def read_raw_with_drift(file_path, entity):
    """Read raw JSONL using an explicit StructType and detect new top-level keys.

    The contract records approved GitHub API fields, including legitimate fields
    that are not used in Bronze/Silver. Only unapproved top-level fields are drift.
    from_json preserves the bad raw line in _corrupt_record for type/JSON errors.
    """
    if entity not in ALLOWED_SOURCE_FIELDS:
        raise ValueError(f"No source field contract for: {entity}")

    allowed = ALLOWED_SOURCE_FIELDS[entity]

    return (
        spark.read.text(file_path)
        .select(F.col("value").alias("original_json"))
        .withColumn(
            "_parsed",
            F.from_json(
                F.col("original_json"),
                RAW_SCHEMAS[entity],
                {
                    "mode": "PERMISSIVE",
                    "columnNameOfCorruptRecord": "_corrupt_record",
                },
            ),
        )
        .withColumn(
            "_unexpected_fields",
            F.array_except(
                F.coalesce(
                    F.expr("json_object_keys(original_json)"),
                    F.expr("CAST(array() AS ARRAY<STRING>)"),
                ),
                F.array(*[F.lit(name) for name in allowed]),
            ),
        )
        .select("original_json", "_unexpected_fields", "_parsed.*")
    )

# ============================================================
# 6. RAW -> BRONZE PROCESSING
# ============================================================

def run_bronze_one(item):

    """

    Read one Raw JSONL file using an explicit schema.

    Steps:

    1. Read Raw data

    2. Validate required fields

    3. Quarantine invalid records

    4. Generate business keys and payload hashes

    5. Deduplicate records

    6. MERGE valid records into Bronze

    7. Log execution results

    """

    repo = item["repository"]

    entity = item["entity"]

    file_path = item["path"]

    started = now()

    execution_id = str(uuid.uuid4())

    count = 0

    t0 = time.perf_counter()

    try:

        # Explicit StructType parsing + unexpected source field detection.
        # Uses text/from_json so Serverless can query _corrupt_record safely.
        raw = read_raw_with_drift(file_path, entity)

        count = raw.count()

        # Determine the logical source identifier

        raw_id = (

            F.col("sha")

            if entity == "commits"

            else F.col("id").cast("string")

        )

        # Choose the source update timestamp

        if entity == "commits":

            updated = F.col("commit.author.date")

        elif entity == "contributors":

            # Contributors have no updated_at field

            updated = F.lit(None).cast("string")

        else:

            # Issues, Pull Requests and Repository Metadata

            updated = F.col("updated_at")

        # ----------------------------------------------------

        # BRONZE VALIDATION

        # ----------------------------------------------------

        reason = (

            F.when(

                F.col("_corrupt_record").isNotNull(),

                "BAD_JSON_OR_TYPE"

            )

            .when(

                F.size(F.col("_unexpected_fields")) > 0,

                "UNEXPECTED_SOURCE_FIELD"

            )

            .when(

                raw_id.isNull() | (F.trim(raw_id) == ""),

                "MISSING_PRIMARY_ID"

            )

            .when(

                F.col("_repo").isNotNull()

                & (F.col("_repo") != repo),

                "REPOSITORY_MISMATCH"

            )

        )

        # Exclude pull requests returned by the Issues endpoint

        if entity == "issues":

            reason = reason.when(

                F.col("pull_request").isNotNull(),

                "PULL_REQUEST_IN_ISSUES"

            )

        # Both snapshot entities require a valid collection time

        elif entity in SNAPSHOT_ENTITIES:

            reason = reason.when(

                F.expr(

                    "try_cast(_ingested_at as timestamp)"

                ).isNull(),

                "INVALID_SNAPSHOT_TIMESTAMP"

            )

        # Attach validation status and source ID

        staged = (

            raw

            .withColumn("_reason", reason)

            .withColumn("_entity_id", raw_id)

            .withColumn("_updated_str", updated)

        )

        # Select all explicitly modeled source fields

        payload_cols = [

            F.col(c)

            for c in RAW_SCHEMAS[entity].fieldNames()

            if c != "_corrupt_record"

        ]

        # ----------------------------------------------------

        # BRONZE QUARANTINE

        # ----------------------------------------------------

        failed = staged.filter(

            F.col("_reason").isNotNull()

        )

        # Preserve the rejected payload for debugging

        rejected_payload = F.coalesce(

            F.col("original_json"),

            F.col("_corrupt_record"),

            F.to_json(F.struct(*payload_cols))

        )

        quarantined_df = failed.select(

            F.sha2(

                F.concat_ws(

                    "|",

                    F.lit(batch_id),

                    F.lit(repo),

                    F.lit(entity),

                    rejected_payload

                ),

                256

            ).alias("quarantine_id"),

            F.lit(batch_id).alias("batch_id"),

            F.lit(repo).alias("repository"),

            F.lit(entity).alias("entity"),

            F.lit(file_path).alias("source_file"),

            rejected_payload.alias("raw_payload"),

            F.col("_reason").alias("error_type"),

            F.concat(

                F.lit("Failed Bronze validation: "),

                F.col("_reason")

            ).alias("error_reason"),

            F.current_timestamp().alias("load_timestamp"),

            F.current_timestamp().alias("quarantined_at")

        )

        # Store rejected records without duplicating quarantines

        quarantined_count = quarantine(

            spark, quarantined_df, "bronze"

        )

        # ----------------------------------------------------

        # PREPARE VALID BRONZE RECORDS

        # ----------------------------------------------------

        valid = staged.filter(

            F.col("_reason").isNull()

        )

        # Hash only business fields for change detection

        business_json = F.to_json(

            F.struct(*[

                F.col(c)

                for c in BUSINESS_COLS[entity]

            ])

        )

        # Preserve the explicitly parsed API payload

        raw_json = F.to_json(

            F.struct(*payload_cols)

        )

        source = valid.select(

            # Logical primary key

            F.concat_ws(

                "|",

                F.lit(repo),

                F.col("_entity_id")

            ).alias("record_key"),

            # Source and ingestion lineage

            F.lit(repo).alias("repository"),

            F.lit(batch_id).alias("batch_id"),

            F.lit(file_path).alias("source_file"),

            # GitHub source-update timestamp

            F.expr(

                "try_cast(_updated_str as timestamp)"

            ).alias("source_updated_at"),

            # Original parsed payload and business hash

            raw_json.alias("raw_payload"),

            F.sha2(business_json, 256).alias("payload_hash"),

            # Pipeline and snapshot timestamps

            F.current_timestamp().alias("load_timestamp"),

            F.expr(

                "try_cast(_ingested_at as timestamp)"

            ).alias("ingested_at")

        )

        # ----------------------------------------------------

        # BRONZE DEDUPLICATION

        # ----------------------------------------------------

        # Snapshot datasets use collection time.

        # Core entities use their GitHub update timestamp.

        order_column = (

            "ingested_at"

            if entity in SNAPSHOT_ENTITIES

            else "source_updated_at"

        )

        ranking = Window.partitionBy(

            "record_key"

        ).orderBy(

            F.col(order_column).desc_nulls_last(),

            F.col("payload_hash").desc()

        )

        # Keep the newest version of each logical key

        source = (

            source

            .withColumn(

                "_rn",

                F.row_number().over(ranking)

            )

            .filter(F.col("_rn") == 1)

            .drop("_rn")

        )

        valid_count = source.count()

        # ----------------------------------------------------

        # BRONZE DELTA MERGE

        # ----------------------------------------------------

        if not valid_count:

            inserted, updated_count = 0, 0

        elif entity in SNAPSHOT_ENTITIES:

            # Repository Metadata and Contributors

            inserted, updated_count = merge_delta_metadata_snapshot(

                source,

                f"{CATALOG}.qa_drift_bronze.{entity}",

                "ingested_at"

            )

        else:

            # Commits, Issues and Pull Requests

            inserted, updated_count = merge_delta_changed(

                source,

                f"{CATALOG}.qa_drift_bronze.{entity}"

            )

        # ----------------------------------------------------

        # BRONZE AUDIT LOG

        # ----------------------------------------------------

        result = {

            "repository": repo,

            "entity": entity,

            "rows_processed": count,

            "valid_keys": valid_count,

            "inserted": inserted,

            "updated": updated_count,

            "upserted": inserted + updated_count,

            "quarantined": quarantined_count

        }

        log(

            spark, execution_id, batch_id,

            "raw_to_bronze", "BRONZE",

            run_date, load_type, started,

            "SUCCESS", count,

            upserted=result["upserted"],

            quarantined=quarantined_count,

            entity_type=entity,

            repository=repo,

            source_file=file_path,

            inserted=inserted,

            updated=updated_count

        )

        print(

            f"BRONZE {repo:30} {entity:13} "

            f"input={count:<6} "

            f"inserted={inserted:<5} "

            f"updated={updated_count:<5} "

            f"quarantine={quarantined_count} "

            f"({time.perf_counter()-t0:.1f}s)",

            flush=True

        )

        # Return Raw file keys so Silver can process them

        # even when Bronze MERGE makes no changes

        # Only keys from accepted records may enter Silver. Without this,
        # a drifted record with an existing key could re-trigger Silver.
        return result, source_keys_from_raw(

            valid, repo, entity

        )

    except Exception:

        # Record the failure before raising the error

        try:

            log(

                spark, execution_id, batch_id,

                "raw_to_bronze", "BRONZE",

                run_date, load_type, started,

                "FAILED", count,

                traceback.format_exc(),

                entity_type=entity,

                repository=repo,

                source_file=file_path

            )

        except Exception as exc:

            print(

                "BRONZE audit failure:",

                repr(exc),

                flush=True

            )

        raise

# ============================================================

# 7. BRONZE -> SILVER PROCESSING

# ============================================================

def run_silver_one(item, current_batch_keys):

    """

    Process accepted Bronze records into typed Silver tables.

    Only records belonging to the current Raw input keys

    are selected for processing.

    """

    repo = item["repository"]

    entity = item["entity"]

    started = now()

    execution_id = str(uuid.uuid4())

    count = 0

    t0 = time.perf_counter()

    try:

        # ----------------------------------------------------

        # READ THE RELEVANT BRONZE RECORDS

        # ----------------------------------------------------

        # Filtering by keys allows idempotent retries.

        # Filtering only by batch_id would miss unchanged rows.

        bronze = (

            spark.table(f"{CATALOG}.qa_drift_bronze.{entity}")

            .filter(F.col("repository") == repo)

            .join(

                current_batch_keys,

                "record_key",

                "left_semi"

            )

        )

        count = bronze.count()

        # Nothing to process for this repository/entity

        if count == 0:

            result = {

                "repository": repo,

                "entity": entity,

                "rows_processed": 0,

                "valid_keys": 0,

                "inserted": 0,

                "updated": 0,

                "upserted": 0,

                "quarantined": 0

            }

            log(

                spark, execution_id, batch_id,

                "bronze_to_silver", "SILVER",

                run_date, load_type, started,

                "SUCCESS", 0,

                entity_type=entity,

                repository=repo,

                source_file=item.get("path"),

                inserted=0,

                updated=0

            )

            print(

                f"SILVER {repo:30} {entity:13} no valid keys "

                f"({time.perf_counter()-t0:.1f}s)",

                flush=True

            )

            return result

        # Parse Bronze raw_payload using the explicit Raw schema

        parsed = bronze.withColumn(

            "api",

            F.from_json(

                "raw_payload",

                RAW_SCHEMAS[entity]

            )

        )

        # Helper for converting ISO timestamp strings

        ts = lambda expr: F.expr(

            f"try_cast({expr} as timestamp)"

        )

        # ----------------------------------------------------

        # SILVER: COMMITS

        # ----------------------------------------------------

        if entity == "commits":

            silver = parsed.select(

                "record_key",

                "repository",

                F.col("api.sha").alias("commit_sha"),

                F.col("api.author.id").alias("author_user_id"),

                ts("api.commit.author.date").alias("commit_timestamp"),

                F.col("api.commit.message").alias("message"),

                "batch_id",

                "source_updated_at",

                "payload_hash",

                F.current_timestamp().alias("load_timestamp")

            )

            reason = (

                F.when(

                    F.col("commit_sha").isNull(),

                    "NULL_COMMIT_SHA"

                )

                .when(

                    F.col("commit_timestamp").isNull(),

                    "INVALID_COMMIT_TIMESTAMP"

                )

            )

        # ----------------------------------------------------

        # SILVER: ISSUES

        # ----------------------------------------------------

        elif entity == "issues":

            silver = parsed.select(

                "record_key",

                "repository",

                F.col("api.id").alias("issue_id"),

                F.col("api.number").alias("issue_number"),

                F.col("api.user.id").alias("author_user_id"),

                F.col("api.title").alias("title"),

                F.col("api.state").alias("state"),

                ts("api.created_at").alias("created_at"),

                ts("api.updated_at").alias("updated_at"),

                ts("api.closed_at").alias("closed_at"),

                "batch_id",

                "source_updated_at",

                "payload_hash",

                F.current_timestamp().alias("load_timestamp")

            )

            reason = (

                F.when(

                    F.col("issue_id").isNull(),

                    "NULL_ISSUE_ID"

                )

                .when(

                    F.col("created_at").isNull()

                    | F.col("updated_at").isNull(),

                    "INVALID_TIMESTAMP"

                )

            )

        # ----------------------------------------------------

        # SILVER: REPOSITORY METADATA

        # ----------------------------------------------------

        elif entity == "repo_metadata":

            # Keep repository metrics for analytics

            silver = parsed.select(

                "record_key",

                "repository",

                F.col("api.id").alias("repo_id"),

                F.col("api.description").alias("description"),

                F.col("api.html_url").alias("html_url"),

                F.col("api.language").alias("language"),

                F.col("api.default_branch").alias("default_branch"),

                F.col("api.stargazers_count").alias("stars_count"),

                F.col("api.forks_count").alias("forks_count"),

                F.col("api.open_issues_count").alias("open_issues_count"),

                F.col("api.subscribers_count").alias("subscribers_count"),

                F.col("api.archived").alias("archived"),

                ts("api.created_at").alias("created_at"),

                ts("api.updated_at").alias("updated_at"),

                ts("api.pushed_at").alias("pushed_at"),

                F.col("ingested_at").alias("snapshot_at"),

                "batch_id",

                "source_updated_at",

                "payload_hash",

                F.current_timestamp().alias("load_timestamp")

            )

            # FIXED: Metadata validation is inside

            # the Repository Metadata branch, not Contributors.

            reason = (

                F.when(

                    F.col("repo_id").isNull(),

                    "NULL_REPOSITORY_ID"

                )

                .when(

                    F.col("created_at").isNull(),

                    "INVALID_CREATED_AT"

                )

                .when(

                    F.col("snapshot_at").isNull(),

                    "INVALID_SNAPSHOT_TIMESTAMP"

                )

            )

        # ----------------------------------------------------

        # SILVER: CONTRIBUTORS

        # ----------------------------------------------------

        elif entity == "contributors":

            # Keep numeric IDs, not usernames/profile URLs.

            silver = parsed.select(

                "record_key",

                "repository",

                F.col("api.id").alias("contributor_id"),

                F.col("api.contributions").alias("contributions"),

                F.col("api.type").alias("account_type"),

                # Snapshot collection timestamp from Bronze

                F.col("ingested_at").alias("snapshot_at"),

                "batch_id",

                "payload_hash",

                F.current_timestamp().alias("load_timestamp")

            )

            # Contributors have their own validation rules

            reason = (

                F.when(

                    F.col("contributor_id").isNull(),

                    "NULL_CONTRIBUTOR_ID"

                )

                .when(

                    F.col("contributions").isNull()

                    | (F.col("contributions") < 0),

                    "INVALID_CONTRIBUTIONS"

                )

                .when(

                    F.col("snapshot_at").isNull(),

                    "INVALID_SNAPSHOT_TIMESTAMP"

                )

            )

        # ----------------------------------------------------

        # SILVER: PULL REQUESTS

        # ----------------------------------------------------

        elif entity == "pulls":

            silver = parsed.select(

                "record_key",

                "repository",

                F.col("api.id").alias("pr_id"),

                F.col("api.number").alias("pr_number"),

                F.col("api.user.id").alias("author_user_id"),

                F.col("api.title").alias("title"),

                F.col("api.state").alias("state"),

                F.col("api.draft").alias("draft"),

                ts("api.created_at").alias("created_at"),

                ts("api.updated_at").alias("updated_at"),

                ts("api.closed_at").alias("closed_at"),

                ts("api.merged_at").alias("merged_at"),

                "batch_id",

                "source_updated_at",

                "payload_hash",

                F.current_timestamp().alias("load_timestamp")

            )

            reason = (

                F.when(

                    F.col("pr_id").isNull(),

                    "NULL_PR_ID"

                )

                .when(

                    F.col("created_at").isNull()

                    | F.col("updated_at").isNull(),

                    "INVALID_TIMESTAMP"

                )

            )

        else:

            # Reject unsupported entities

            raise ValueError(

                f"Unknown entity for Silver: {entity}"

            )

        # ----------------------------------------------------

        # SILVER QUARANTINE

        # ----------------------------------------------------

        # Mark records failing Silver validation

        staged = silver.withColumn("_reason", reason)

        failed = staged.filter(

            F.col("_reason").isNotNull()

        )

        # Join the original Bronze payload for debugging

        q = (

            failed

            .join(

                bronze.select("record_key", "raw_payload"),

                "record_key",

                "left"

            )

            .select(

                F.sha2(

                    F.concat_ws(

                        "|",

                        F.lit(batch_id),

                        F.lit(entity),

                        "record_key",

                        "_reason"

                    ),

                    256

                ).alias("quarantine_id"),

                F.lit(batch_id).alias("batch_id"),

                F.col("repository"),

                F.lit(entity).alias("entity"),

                F.col("record_key"),

                F.col("raw_payload"),

                F.col("_reason").alias("error_type"),

                F.concat(

                    F.lit("Silver validation failure: "),

                    F.col("_reason")

                ).alias("error_reason"),

                F.current_timestamp().alias("load_timestamp"),

                F.current_timestamp().alias("quarantined_at")

            )

        )

        quarantined_count = quarantine(

            spark, q, "silver"

        )

        # Continue only with valid records

        valid = (

            staged

            .filter(F.col("_reason").isNull())

            .drop("_reason")

        )

        # ----------------------------------------------------

        # SILVER DEDUPLICATION

        # ----------------------------------------------------

        # Contributors and metadata use snapshot timestamps.

        # Other entities use source_updated_at.

        order_column = (

            "snapshot_at"

            if entity in SNAPSHOT_ENTITIES

            else "source_updated_at"

        )

        ranking = Window.partitionBy(

            "record_key"

        ).orderBy(

            F.col(order_column).desc_nulls_last(),

            F.col("payload_hash").desc()

        )

        valid = (

            valid

            .withColumn(

                "_rn",

                F.row_number().over(ranking)

            )

            .filter(F.col("_rn") == 1)

            .drop("_rn")

        )

        valid_count = valid.count()

        # ----------------------------------------------------

        # SILVER DELTA MERGE

        # ----------------------------------------------------

        if not valid_count:

            inserted, updated_count = 0, 0

        elif entity in SNAPSHOT_ENTITIES:

            # Snapshot-aware MERGE for Contributors and Metadata

            inserted, updated_count = merge_delta_metadata_snapshot(

                valid,

                f"{CATALOG}.qa_drift_silver.{entity}",

                "snapshot_at"

            )

        else:

            # Timestamp-based MERGE for core entities

            inserted, updated_count = merge_delta_changed(

                valid,

                f"{CATALOG}.qa_drift_silver.{entity}"

            )

        # ----------------------------------------------------

        # SILVER AUDIT LOG

        # ----------------------------------------------------

        result = {

            "repository": repo,

            "entity": entity,

            "rows_processed": count,

            "valid_keys": valid_count,

            "inserted": inserted,

            "updated": updated_count,

            "upserted": inserted + updated_count,

            "quarantined": quarantined_count

        }

        log(

            spark, execution_id, batch_id,

            "bronze_to_silver", "SILVER",

            run_date, load_type, started,

            "SUCCESS", count,

            upserted=result["upserted"],

            quarantined=quarantined_count,

            entity_type=entity,

            repository=repo,

            source_file=item.get("path"),

            inserted=inserted,

            updated=updated_count

        )

        print(

            f"SILVER {repo:30} {entity:13} "

            f"input={count:<6} "

            f"inserted={inserted:<5} "

            f"updated={updated_count:<5} "

            f"quarantine={quarantined_count} "

            f"({time.perf_counter()-t0:.1f}s)",

            flush=True

        )

        return result

    except Exception:

        # Record the error in the execution log

        try:

            log(

                spark, execution_id, batch_id,

                "bronze_to_silver", "SILVER",

                run_date, load_type, started,

                "FAILED", count,

                traceback.format_exc(),

                entity_type=entity,

                repository=repo,

                source_file=item.get("path")

            )

        except Exception as exc:

            print(

                "SILVER audit failure:",

                repr(exc),

                flush=True

            )

        raise


In [0]:
# Execute the SAME updated processing functions on QA tables.
# Deliberately: 2 valid issues, 1 drifted issue, 1 bad ID, 1 invalid Silver timestamp.
# The approved GitHub comments_url field must be allowed.

def qa_drift_integration(load_mode):
    global batch_id, run_date, load_type
    assert load_mode in ("FULL", "INCREMENTAL")
    assert "comments_url" in ALLOWED_SOURCE_FIELDS["issues"]

    repo = "apache/spark"
    entity = "issues"
    batch_id = f"qa_drift_{load_mode.lower()}_{uuid.uuid4().hex[:12]}"
    run_date = datetime.now(timezone.utc).date().isoformat()
    load_type = load_mode
    seed = (uuid.uuid4().int % 100000000000) + 8000000000000
    base = {
        "number": 17,
        "title": "QA synthetic issue",
        "state": "open",
        "created_at": "2026-10-08T10:00:00Z",
        "updated_at": "2026-10-08T11:00:00Z",
        "_repo": repo,
        "_ingested_at": "2026-10-09T00:00:00Z",
        "_batch_id": batch_id,
    }
    def record(**overrides):
        return {**base, **overrides}

    input_rows = [
        record(id=seed),  # VALID
        record(id=seed+1, comments_url="https://api.github.com/qa/known-field"),  # VALID; known unmodeled field
        record(id=seed+2, brand_new_field_qa="UNEXPECTED"),  # QUARANTINE: drift
        record(id="not-a-number"),  # QUARANTINE: bad type
        record(id=seed+3, created_at="not-a-date"),  # Bronze valid; Silver QUARANTINE
    ]
    path = QA_ROOT / batch_id / "synthetic_issues.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(x) for x in input_rows) + "\n", encoding="utf-8")
    item = {"repository": repo, "entity": entity, "path": str(path)}

    print(f"\n========== {load_mode} QA INTEGRATION ==========", flush=True)
    print("Batch:", batch_id)
    prod_bronze_before = spark.table(f"{CATALOG}.bronze.issues").count()
    prod_silver_before = spark.table(f"{CATALOG}.silver.issues").count()

    # These functions are the production Cell 5 definitions, with table destination names redirected.
    bronze_result, keys = run_bronze_one(item)
    silver_result = run_silver_one(item, keys)

    assert bronze_result["rows_processed"] == 5, bronze_result
    assert bronze_result["valid_keys"] == 3, bronze_result
    assert bronze_result["quarantined"] == 2, bronze_result
    assert silver_result["rows_processed"] == 3, silver_result
    assert silver_result["valid_keys"] == 2, silver_result
    assert silver_result["quarantined"] == 1, silver_result

    bq = spark.table(QA_BRONZE_Q).filter(F.col("batch_id") == batch_id)
    sq = spark.table(QA_SILVER_Q).filter(F.col("batch_id") == batch_id)
    b = spark.table(QA_BRONZE).filter(F.col("batch_id") == batch_id)
    s = spark.table(QA_SILVER).filter(F.col("batch_id") == batch_id)
    assert bq.count() == 2, "Expected 2 Bronze quarantines"
    assert sq.count() == 1, "Expected 1 Silver quarantine"
    assert b.count() == 3, "Expected 3 accepted Bronze issues"
    assert s.count() == 2, "Expected 2 accepted Silver issues"

    bronze_reasons = {x.error_type for x in bq.select("error_type").collect()}
    silver_reasons = {x.error_type for x in sq.select("error_type").collect()}
    assert bronze_reasons == {"BAD_JSON_OR_TYPE", "UNEXPECTED_SOURCE_FIELD"}, bronze_reasons
    assert silver_reasons == {"INVALID_TIMESTAMP"}, silver_reasons

    # Original JSON for drifted record must retain the new field.
    drift_payloads = [x.raw_payload for x in bq.filter(F.col("error_type") == "UNEXPECTED_SOURCE_FIELD")
                      .select("raw_payload").collect()]
    assert len(drift_payloads) == 1 and "brand_new_field_qa" in drift_payloads[0]

    # Re-run identical input: no new inserts/updates or duplicate quarantine rows.
    bronze_again, keys_again = run_bronze_one(item)
    silver_again = run_silver_one(item, keys_again)
    assert bronze_again["upserted"] == 0, bronze_again
    assert silver_again["upserted"] == 0, silver_again
    assert bq.count() == 2 and sq.count() == 1, "Quarantine grew on unchanged replay"
    assert b.count() == 3 and s.count() == 2, "Bronze/Silver duplicated on replay"
    assert spark.table(f"{CATALOG}.bronze.issues").count() == prod_bronze_before
    assert spark.table(f"{CATALOG}.silver.issues").count() == prod_silver_before

    # We expect two successful file logs per layer (initial run + replay).
    audit = spark.table(LOG_TABLE).filter(F.col("batch_id") == batch_id)
    assert audit.filter(F.col("layer") == "BRONZE").count() == 2
    assert audit.filter(F.col("layer") == "SILVER").count() == 2
    assert audit.filter(F.col("status") != "SUCCESS").count() == 0

    print(f"PASS: {load_mode} — Bronze 3 accepted / 2 quarantined")
    print(f"PASS: {load_mode} — Silver 2 accepted / 1 quarantined")
    print(f"PASS: {load_mode} — unexpected field preserved; known field accepted")
    print(f"PASS: {load_mode} — replay upserts=0 and no duplicates")
    print(f"PASS: {load_mode} — QA audit logged success; production counts unchanged")
    return batch_id


In [0]:
full_qa_batch = qa_drift_integration("FULL")

In [0]:
incremental_qa_batch = qa_drift_integration("INCREMENTAL")